# Данные: оптические стёкла
Источники: **SciGlass** (glasspy) + **SCHOTT** каталог (PDF)

In [ ]:
import sys
sys.path.insert(0, 'C:/Users/user/AppData/Roaming/Python/Python314/site-packages')
import numpy as np
import pandas as pd
print('OK')

## 1. SciGlass — свойства (~422k стёкол)

In [ ]:
import zipfile, io
from pathlib import Path

DATA_DIR = Path('C:/Users/user/AppData/Local/GlassPy/GlassPy/data')

# Свойства: n, v, Tg, плотность, КТР и др.
with zipfile.ZipFile(DATA_DIR / 'select_SciGK.csv.zip') as z:
    with z.open(z.namelist()[0]) as f:
        df_props = pd.read_csv(f, sep='\t', low_memory=False)

df_props['ID'] = df_props['KOD'] * 100_000_000 + df_props['GLASNO']
df_props = df_props.drop_duplicates('ID').set_index('ID')

print(f'Свойства: {df_props.shape[0]:,} записей, {df_props.shape[1]} колонок')
print('Колонки:', df_props.columns.tolist())

In [ ]:
# Переименовываем нужные колонки
# Смотрим translators glasspy для маппинга
import sys; sys.path.insert(0,'C:/Users/user/AppData/Roaming/Python/Python314/site-packages')
from glasspy.data.translators import SciGK_translation

rename = {k: v['rename'] for k, v in SciGK_translation.items() if 'rename' in v}
df_props = df_props.rename(columns=rename)

# Смотрим заполненность ключевых оптических свойств
key_props = ['RefractiveIndex', 'AbbeNumber', 'Density', 'TransitionTemperature', 'CTE']
for p in key_props:
    if p in df_props.columns:
        nn = df_props[p].replace(0, np.nan).dropna()
        print(f'{p:30s}: {len(nn):>8,}')

## 2. SciGlass — элементный состав

In [ ]:
with zipfile.ZipFile(DATA_DIR / 'select_AtMol.csv.zip') as z:
    with z.open(z.namelist()[0]) as f:
        df_elem = pd.read_csv(f, sep='\t', low_memory=False)

df_elem['ID'] = df_elem['Kod'] * 100_000_000 + df_elem['GlasNo']
df_elem = df_elem.drop_duplicates('ID').set_index('ID').drop(columns=['Kod','GlasNo'], errors='ignore')

print(f'Элементный состав: {df_elem.shape[0]:,} записей, {df_elem.shape[1]} элементов')
print('Элементы:', df_elem.columns.tolist()[:30], '...')

## 3. SciGlass — состав в оксидах
> Осторожно: 726 оксидов x 422k строк ~ 2.3 GB. Загружаем только для стёкол с оптическими свойствами.

In [ ]:
# Сначала отбираем ID стёкол с n + v
has_n = df_props['RefractiveIndex'].replace(0, np.nan).notna()
has_v = df_props['AbbeNumber'].replace(0, np.nan).notna()
opt_ids = set(df_props[has_n & has_v].index)
print(f'Стёкол с n и v: {len(opt_ids):,}')

In [ ]:
# Парсим состав только для нужных ID
records = []
with zipfile.ZipFile(DATA_DIR / 'select_Gcomp.csv.zip') as z:
    with z.open(z.namelist()[0]) as f:
        raw = pd.read_csv(f, sep='\t', low_memory=False)

raw['ID'] = raw['Kod'] * 100_000_000 + raw['GlasNo']
raw = raw[raw['ID'].isin(opt_ids)].drop_duplicates('ID').set_index('ID')

# Парсим строку состава вида |SiO2|mol|61.5|wt|...
def parse_comp(s):
    if not isinstance(s, str): return {}
    parts = s.strip('|').split('\x7f')
    return {parts[n*4]: float(parts[n*4+3]) for n in range(len(parts)//4)
            if len(parts) > n*4+3 and parts[n*4+3].replace('.','').lstrip('-').isdigit()}

comp_list = [parse_comp(row) for row in raw['Composition'].values]
df_comp = pd.DataFrame(comp_list, index=raw.index).fillna(0)
print(f'Состав (оксиды): {df_comp.shape[0]:,} стёкол x {df_comp.shape[1]} компонентов')

## 4. Объединённый датасет SciGlass

In [ ]:
df_sciglass = df_props.join(df_elem, how='left', rsuffix='_at').join(df_comp, how='left')
df_sciglass = df_sciglass[
    df_sciglass['RefractiveIndex'].replace(0,np.nan).notna() &
    df_sciglass['AbbeNumber'].replace(0,np.nan).notna()
]
print(f'SciGlass итог: {df_sciglass.shape[0]:,} записей x {df_sciglass.shape[1]} колонок')
df_sciglass[['RefractiveIndex','AbbeNumber','Density','TransitionTemperature']].describe().round(3)

In [ ]:
df_sciglass.to_parquet('sciglass_optical.parquet')
print('Сохранено: sciglass_optical.parquet')

## 5. SCHOTT каталог (PDF)
nd, vd, плотность, Tg коммерческих марок. Состав не публикуется.

In [ ]:
from pypdf import PdfReader
import re

reader = PdfReader('schott_catalog.pdf')
text = ''.join(p.extract_text() or '' for p in reader.pages)
print(f'Страниц: {len(reader.pages)}, символов: {len(text):,}')
print(text[:1000])

In [ ]:
# Ищем строки вида: N-BK7  1.5168  64.17  2.51  557
# Паттерн: марка  nd   vd   density  Tg
pattern = r'([A-Z][A-Z0-9\-]+)\s+(1\.\d{3,4})\s+(\d{2,3}\.\d{1,2})\s+(\d\.\d{2})\s+(\d{3,4})'
rows = []
for m in re.finditer(pattern, text):
    rows.append({
        'glass': m.group(1),
        'nd': float(m.group(2)),
        'vd': float(m.group(3)),
        'density': float(m.group(4)),
        'Tg': float(m.group(5)),
    })

df_schott = pd.DataFrame(rows).drop_duplicates('glass')
print(f'SCHOTT: {len(df_schott)} марок извлечено')
df_schott.head(10)

In [ ]:
df_schott.to_csv('schott_glasses.csv', index=False)
print('Сохранено: schott_glasses.csv')

## Итог

In [ ]:
print('=== Загруженные данные ===')
print(f'SciGlass (sciglass_optical.parquet): {len(df_sciglass):,} стёкол')
print(f'SCHOTT   (schott_glasses.csv)       : {len(df_schott)} марок')
print()
print('Ключевые свойства в SciGlass:')
for p in ['RefractiveIndex','AbbeNumber','Density','TransitionTemperature','CTE']:
    if p in df_sciglass.columns:
        nn = df_sciglass[p].replace(0,np.nan).dropna()
        print(f'  {p:30s}: {len(nn):,}')